# aule — Automatic validation report

`aule.report.generate_report` runs a curated set of metrics and plots on a `(y_true, y_pred)` pair and assembles everything into a single self-contained HTML file — no external assets, all figures embedded as base64 PNGs — so you get a comprehensive quick-look validation summary without calling each function by hand.

In [ ]:
!pip install aule

## Basic usage

The simplest case: just pass ground truth, prediction, and an output path.

In [ ]:
import numpy as np
from aule.report import generate_report

np.random.seed(20)

gt   = np.random.rand(8, 64, 64, 1)
pred = gt + np.random.normal(0, 0.1, gt.shape)

path = generate_report(gt, pred, save_path='report.html')
print('Report written to:', path)

## What's inside

The report includes a metrics table (RMSE, MAE, MSE, bias, Pearson r, R2, SSIM, PSNR, MAPE, SMAPE, KGE, max error, explained variance, spectral error, gradient error, PSD radial error, Wasserstein distance, quantile mapping bias, plus pixelwise correlation when there's a meaningful sample dimension) and a gallery of plots (scatter, histogram comparison, CDF comparison, error histogram, field comparison, error map, box plot, Taylor diagram).

If the input has a meaningful time axis, climate-specific metrics and a temporal trend plot are added automatically.

In [ ]:
# with a time axis: climate diagnostics get included automatically
gt_t   = np.random.rand(16, 16, 1, 30)
pred_t = gt_t + np.random.normal(0, 0.1, gt_t.shape)

path = generate_report(gt_t, pred_t, save_path='report_with_time.html', data_format='hwct',
                        title='Climate model validation report')
print('Report written to:', path)

## Handling missing data

Like every other aule function, `ignore_nan=True` excludes non-finite values wherever the underlying metric/plot supports it.

In [ ]:
gt_with_gaps = gt.copy()
gt_with_gaps[:, :5, :5] = np.nan

path = generate_report(gt_with_gaps, pred, save_path='report_with_nan.html', ignore_nan=True)
print('Report written to:', path)

## Inspecting the report inline

Since the report is just an HTML file, you can preview it directly inside a notebook with an IFrame, in addition to opening it in a browser.

In [ ]:
from IPython.display import IFrame

IFrame(src='report.html', width=900, height=600)

## When functions don't apply

If a metric isn't applicable to the given data (for example, a climate metric called without a meaningful time axis raising an internal error), `generate_report` skips it gracefully and shows `n/a` with a short reason in the table, rather than crashing the whole report.